# 第5课 AI魔法师

适合对象：完成第1~4课的同学

前置知识：
- 会用模型做图片识别
- 了解图像预处理和模型预测流程

学习目标：
- 做一个“识别 + 语音播报”的小功能
- 体验“风格迁移”效果（把图片变成不同艺术风格）
- 体验 AIGC 图像生成（在线 API 或离线替代方案）


## 课程流程

1. 载入模型并预测图片类别
2. 把识别结果用 TTS 朗读出来
3. 做两种风格迁移效果
4. 进行 AIGC 图像生成
5. 课堂挑战


In [ ]:
# 第1步：准备基础库和识别模型
from pathlib import Path
import os
import base64

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter, ImageEnhance, ImageDraw

try:
    import torch
    from torch import nn
    from torchvision import transforms
except Exception as e:
    raise ImportError(
        '本课需要 PyTorch 与 torchvision。请先安装：pip install torch torchvision'
    ) from e

class SmallCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt_path = Path('../models/lesson03_cnn.pth')
class_names = ['双子塔', '上海科技馆', '张江药谷']

if ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location='cpu')
    class_names = checkpoint.get('class_names', class_names)
    model = SmallCNN(num_classes=len(class_names))
    model.load_state_dict(checkpoint['model_state_dict'])
    print('已加载模型:', ckpt_path.resolve())
else:
    model = SmallCNN(num_classes=len(class_names))
    print('未找到训练模型，将使用随机参数演示流程。建议先完成第3课。')

model = model.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def predict_image(img: Image.Image):
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    idx = int(np.argmax(probs))
    name = class_names[idx] if idx < len(class_names) else f'类别{idx}'
    conf = float(probs[idx])
    return name, conf, probs


## Step 1 识别 + TTS（语音播报）

思路：
- 先预测图片是什么
- 再把结果拼成一句话
- 用 TTS 引擎读出来

如果电脑没有安装 TTS 依赖，也没关系，至少会打印文字结果。


In [ ]:
# 第2步：读取图片并识别
img_candidates = [
    Path('../test.jpg'),
    Path('../result_sketch.jpg'),
    Path('images/sample_building.jpg'),
]
img_path = next((p for p in img_candidates if p.exists()), None)

if img_path is not None:
    input_img = Image.open(img_path).convert('RGB')
    print('输入图片:', img_path)
else:
    # 没有图片时，自动画一张简单示例图
    canvas = np.zeros((240, 320, 3), dtype=np.uint8)
    canvas[..., 1] = np.linspace(40, 140, 240, dtype=np.uint8)[:, None]
    canvas[60:220, 100:220] = [220, 220, 230]
    input_img = Image.fromarray(canvas)
    print('未找到输入图片，已自动生成示例图。')

pred_name, pred_conf, probs = predict_image(input_img)
result_text = f'识别结果：这张图片最像 {pred_name}，置信度 {pred_conf:.1%}。'
print(result_text)

plt.figure(figsize=(5, 4))
plt.imshow(input_img)
plt.title(result_text)
plt.axis('off')
plt.show()

# 尝试语音播报（pyttsx3 是离线 TTS）
try:
    import pyttsx3
    engine = pyttsx3.init()
    engine.say(result_text)
    engine.runAndWait()
    print('TTS 播报完成。')
except Exception as e:
    print('TTS 未执行（通常是未安装 pyttsx3 或系统语音环境缺失）。')
    print('你可以安装后再试：pip install pyttsx3')

# 额外保存文字结果，方便做项目记录
output_dir = Path('../outputs')
output_dir.mkdir(parents=True, exist_ok=True)
text_path = output_dir / 'lesson05_recognition.txt'
text_path.write_text(result_text, encoding='utf-8')
print('识别文本已保存到:', text_path.resolve())


## Step 2 风格迁移（教学版）

真正的神经风格迁移会用到更复杂模型；
本课先做“可运行、易理解”的教学版：
- 素描风（Sketch）
- 赛博风（Cyber）

这样你能先理解“风格变化”的核心思想。


In [ ]:
# 第3步：实现两种风格效果
def sketch_style(img: Image.Image) -> Image.Image:
    # 素描风：灰度 -> 反相 -> 高斯模糊 -> 颜色减淡融合
    gray = ImageOps.grayscale(img)
    invert = ImageOps.invert(gray)
    blur = invert.filter(ImageFilter.GaussianBlur(radius=8))

    # 颜色减淡（dodge）公式：result = base * 255 / (255 - blend)
    base = np.array(gray).astype(np.float32)
    blend = np.array(blur).astype(np.float32)
    dodge = np.clip(base * 255.0 / (255.0 - blend + 1e-6), 0, 255).astype(np.uint8)
    return Image.fromarray(dodge)

def cyber_style(img: Image.Image) -> Image.Image:
    # 赛博风：增强对比度与饱和度，并叠加边缘线条
    color_boost = ImageEnhance.Color(img).enhance(1.8)
    contrast_boost = ImageEnhance.Contrast(color_boost).enhance(1.4)
    edges = contrast_boost.filter(ImageFilter.FIND_EDGES)
    edges = ImageEnhance.Brightness(edges).enhance(1.6)

    # 把边缘按 30% 透明度叠加到原图上
    merged = Image.blend(contrast_boost, edges, alpha=0.3)
    return merged

img_sketch = sketch_style(input_img)
img_cyber = cyber_style(input_img)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(input_img)
plt.title('原图')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img_sketch, cmap='gray')
plt.title('素描风')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_cyber)
plt.title('赛博风')
plt.axis('off')

plt.tight_layout()
plt.show()

output_dir = Path('../outputs')
output_dir.mkdir(parents=True, exist_ok=True)
sketch_path = output_dir / 'lesson05_style_sketch.jpg'
cyber_path = output_dir / 'lesson05_style_cyber.jpg'
img_sketch.save(sketch_path)
img_cyber.save(cyber_path)
print('风格图已保存:', sketch_path.resolve())
print('风格图已保存:', cyber_path.resolve())


## Step 3 AIGC 生成

这里提供两条路径：
- 在线方式：如果你配置了 `OPENAI_API_KEY`，可调用图像模型生成
- 离线方式：如果没有 API Key，就自动生成一张“概念海报占位图”

这样每位同学都能完成本课，不会卡住。


In [ ]:
# 第4步：尝试 AIGC 图像生成
prompt = '未来感的张江科学城，儿童插画风格，明亮色彩，科技感建筑'
output_dir = Path('../outputs')
output_dir.mkdir(parents=True, exist_ok=True)
aigc_path = output_dir / 'lesson05_aigc_generated.png'

api_key = os.getenv('OPENAI_API_KEY', '')

if api_key:
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        result = client.images.generate(
            model='gpt-image-1',
            prompt=prompt,
            size='1024x1024',
        )

        # OpenAI Images 常见返回是 base64 图片数据
        image_b64 = result.data[0].b64_json
        image_bytes = base64.b64decode(image_b64)
        aigc_path.write_bytes(image_bytes)
        print('在线 AIGC 生成成功:', aigc_path.resolve())
    except Exception as e:
        print('在线 AIGC 失败，转为离线占位方案。错误信息:', e)
        api_key = ''  # 触发下面的离线分支

if not api_key:
    # 离线占位方案：自动画一张“未来城市概念图”
    w, h = 1024, 1024
    canvas = Image.new('RGB', (w, h), (18, 28, 58))
    draw = ImageDraw.Draw(canvas)

    # 画渐变天空
    for y in range(h):
        r = int(18 + 40 * y / h)
        g = int(28 + 70 * y / h)
        b = int(58 + 120 * y / h)
        draw.line([(0, y), (w, y)], fill=(r, g, b))

    # 画几栋“科技大楼”
    for x in [180, 360, 560, 760]:
        top = np.random.randint(260, 500)
        draw.rectangle((x, top, x + 140, 920), fill=(40, 55, 95), outline=(120, 220, 255), width=3)
        for wy in range(top + 20, 900, 40):
            for wx in range(x + 15, x + 125, 28):
                draw.rectangle((wx, wy, wx + 12, wy + 18), fill=(170, 240, 255))

    draw.text((40, 40), 'AIGC OFFLINE DEMO', fill=(255, 255, 255))
    draw.text((40, 84), prompt, fill=(220, 240, 255))
    canvas.save(aigc_path)
    print('离线 AIGC 占位图已生成:', aigc_path.resolve())

# 展示 AIGC 结果
show_aigc = Image.open(aigc_path)
plt.figure(figsize=(6, 6))
plt.imshow(show_aigc)
plt.title('AIGC 生成结果')
plt.axis('off')
plt.show()


## 课堂挑战

1. 做一个“输入图片 -> 识别 -> 语音播报 -> 风格化保存”的完整小程序。
2. 自己写 3 条 AIGC 提示词（prompt），比较生成风格差异。
3. 思考：AIGC 生成内容为什么还需要人工审核？

项目扩展建议：
- 给每张结果图加时间戳和作者名
- 把识别结果、置信度、生成图路径写入一个 CSV 日志


In [ ]:
# 挑战脚手架：批量处理一个文件夹中的图片（识别 + 风格化）
def process_folder(folder: str):
    folder_path = Path(folder)
    if not folder_path.exists():
        print('文件夹不存在:', folder_path)
        return

    out_dir = Path('../outputs/batch_results')
    out_dir.mkdir(parents=True, exist_ok=True)

    image_files = sorted([p for p in folder_path.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
    if not image_files:
        print('没有找到可处理图片。')
        return

    for p in image_files:
        img = Image.open(p).convert('RGB')
        name, conf, _ = predict_image(img)
        styled = cyber_style(img)
        save_path = out_dir / f'{p.stem}_cyber_{name}.jpg'
        styled.save(save_path)
        print(f'{p.name} -> {name} ({conf:.1%}) -> 保存: {save_path.name}')

# 使用示例（先准备一个图片文件夹再取消注释）
# process_folder('../demo_images')
